In [ ]:
"""
==============================================================================
RETAIL CREDIT PROJECT (BRAZIL) - STAGE 1: DATA PREPARATION & VIF DIAGNOSTICS
Module responsible for data ingestion, handling localized date formats,
computing descriptive statistics, and Variance Inflation Factor (VIF).
==============================================================================
"""

import logging
from typing import Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings

# ==============================================================================
# GENERAL SETTINGS
# ==============================================================================
warnings.filterwarnings("ignore")

# Logging Configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

# Matplotlib Visual Settings
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#4a6175"

# Global Constants
PT_MONTHS = {
    'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04', 'mai': '05', 'jun': '06',
    'jul': '07', 'ago': '08', 'set': '09', 'out': '10', 'nov': '11', 'dez': '12'
}

EXCLUDED_MODALITIES = [
    "Financiamentos à exportação", 
    "Financiamentos à importação",
    "Financiamentos de infraestrutura e desenvolvimento",
    "Financiamentos de títulos e valores mobiliários",
    "Financiamentos com interveniência",
]

# ==============================================================================
# HELPER AND DATA ENGINEERING FUNCTIONS
# ==============================================================================
def remove_chart_spines(ax: Optional[plt.Axes] = None) -> None:
    """Removes the top, right, left, and bottom spines from a Matplotlib axis."""
    if ax is None:
        ax = plt.gca()
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

def standardize_local_dates(date_str: str) -> str:
    """Converts atypical Brazilian central bank dates (e.g., 'jul/86') to 'MM/YYYY'."""
    clean_date = str(date_str).lower().strip()
    for pt_month, num_month in PT_MONTHS.items():
        if clean_date.startswith(pt_month):
            parts = clean_date.split('/')
            if len(parts) > 1:
                year = parts[1]
                if len(year) == 2:
                    year = f"19{year}" if int(year) > 50 else f"20{year}"
                return f"{num_month}/{year}"
    return clean_date

def load_local_csv(filename: str, target_column: str) -> pd.DataFrame:
    """Loads a time series from a local CSV, parsing localized dates and numerical values."""
    try:
        df = pd.read_csv(f"{filename}.csv", sep=";", decimal=",", encoding="latin-1")
    except FileNotFoundError as e:
        logger.error(f"File '{filename}.csv' not found.")
        raise e
        
    df.columns = ['date', 'value']
    df = df[df['date'].astype(str).str.contains(r'/', na=False)].copy()
    
    df['date'] = df['date'].apply(standardize_local_dates)
    df['date'] = pd.to_datetime(df['date'], format='%m/%Y', errors='coerce')
    
    if df['value'].dtype == 'O':
        df['value'] = df['value'].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        
    df[target_column] = pd.to_numeric(df['value'], errors='coerce')
    df = df.dropna(subset=['date', target_column])
    df['year_month'] = df['date'].dt.to_period('M')
    
    df_res = df[['year_month', target_column]].drop_duplicates(subset=['year_month'])
    logger.info(f"Series '{target_column}' loaded successfully ({len(df_res)} records).")
    return df_res

def map_macro_block_vectorized(df: pd.DataFrame) -> np.ndarray:
    """Maps detailed credit sub-modalities into 4 analytical macro-blocks."""
    sub = df["submodalidade"].astype(str).str.lower()
    mod = df["modalidade"].astype(str).str.lower()

    cond_revolving = sub.str.contains(
        "crédito rotativo vinculado a cartão|rotativo|cheque especial|sem consignação|"
        "adiantamentos a depositantes|conta garantida|fatura parcelada|saque financiado|"
        "antecipação de fatura", regex=True, na=False
    )
    cond_payroll = sub.str.contains("com consignação|consignado", regex=True, na=False)
    cond_collateral = (
        sub.str.contains(
            "habitacional|sfh|veículos automotores|veículos autom.|home equity|"
            "veiculo|veículo|arrendamento financeiro de veículos", regex=True, na=False
        ) | mod.str.contains("imobiliários", regex=True, na=False)
    )

    choices = [
        "1. Revolving / Unsecured",
        "2. Payroll / Secured",
        "3. Real Collateral (Mortgage/Auto)"
    ]
    conditions = [cond_revolving, cond_payroll, cond_collateral]
    
    return np.select(conditions, choices, default="4. Others / Renegotiated")

# ==============================================================================
# MAIN PIPELINE
# ==============================================================================
def main():
    logger.info("STARTING STAGE 1: MACROECONOMIC EXTRACTION AND FEATURE ENGINEERING")
    
    # 1. Macroeconomic Dataset
    df_selic = load_local_csv("selic", "selic_avg")
    df_rates = load_local_csv("juros_pf", "retail_rate")
    df_debt = load_local_csv("endividamento_familias", "total_household_debt")

    df_macro = df_selic.merge(df_rates, on="year_month", how="inner")
    df_macro = df_macro.merge(df_debt, on="year_month", how="inner").sort_values("year_month").reset_index(drop=True)

    df_macro["bank_spread"] = df_macro["retail_rate"] - df_macro["selic_avg"]
    df_macro["selic_lag6"] = df_macro["selic_avg"].shift(6)
    df_macro["household_debt_lag6"] = df_macro["total_household_debt"].shift(6)
    df_macro = df_macro.dropna().reset_index(drop=True)

    # 2. Microdata Panel (Credit Registry)
    logger.info("LOADING AND CONSOLIDATING RETAIL CREDIT PANEL")
    cols_load = [
        "data_base", "modalidade", "submodalidade", "carteira_ativa",
        "vencido_de_15_ate_90_dias", "vencido_acima_de_90_dias", "ativo_problematico"
    ]
    
    try:
        df_raw = pd.read_parquet("painel_scr_pf_bruto.parquet", columns=cols_load)
    except FileNotFoundError as e:
        logger.error("File 'painel_scr_pf_bruto.parquet' not found.")
        raise e

    df_raw = df_raw[~df_raw["modalidade"].isin(EXCLUDED_MODALITIES)]
    df_raw["macro_block"] = map_macro_block_vectorized(df_raw)

    df_panel = df_raw.groupby(["data_base", "macro_block"]).agg(
        active_portfolio=("carteira_ativa", "sum"),
        defaulted_over_90d=("vencido_acima_de_90_dias", "sum"),
        problem_assets=("ativo_problematico", "sum")
    ).reset_index()

    df_panel["default_rate_pct"] = np.where(
        df_panel["active_portfolio"] > 0,
        (df_panel["defaulted_over_90d"] / df_panel["active_portfolio"]) * 100, 0.0
    )
    df_panel["problem_assets_rate_pct"] = np.where(
        df_panel["active_portfolio"] > 0,
        (df_panel["problem_assets"] / df_panel["active_portfolio"]) * 100, 0.0
    )

    # 3. Descriptive Statistics
    logger.info("GENERATING DESCRIPTIVE STATISTICS BY MACRO-BLOCK")
    stats = df_panel.groupby("macro_block").agg(
        avg_balance_bn=("active_portfolio", lambda x: x.mean() / 1e9),
        avg_default=("default_rate_pct", "mean"),
        std_default=("default_rate_pct", "std"),
        min_default=("default_rate_pct", "min"),
        median_default=("default_rate_pct", "median"),
        max_default=("default_rate_pct", "max"),
        avg_prob_assets=("problem_assets_rate_pct", "mean")
    ).reset_index()

    total_balance = stats["avg_balance_bn"].sum()
    stats["avg_share_pct"] = (stats["avg_balance_bn"] / total_balance) * 100

    stats = stats[[
        "macro_block", "avg_balance_bn", "avg_share_pct", "avg_default",
        "std_default", "min_default", "median_default", "max_default", "avg_prob_assets"
    ]].sort_values("avg_balance_bn", ascending=False)

    print("\n" + "="*80)
    print("GLOBAL DESCRIPTIVE STATISTICS:")
    print("="*80)
    print(stats.to_string(index=False))
    print("="*80 + "\n")

    # 4. Multicollinearity and VIF Diagnostics
    logger.info("COMPUTING CORRELATION MATRIX AND VIF")
    macro_test_cols = [
        "selic_avg", "selic_lag6", "bank_spread", 
        "total_household_debt", "household_debt_lag6"
    ]

    vif_plot_names = {
        "selic_avg": "Selic Rate (% p.m.)",
        "selic_lag6": "Selic (6m Lag)",
        "bank_spread": "Bank Spread",
        "total_household_debt": "Total Household Debt",
        "household_debt_lag6": "Household Debt (6m Lag)"
    }

    X_vif = df_macro[macro_test_cols].copy()
    X_vif_const = sm.add_constant(X_vif)

    vif_data = pd.DataFrame()
    vif_data["Variable"] = [vif_plot_names.get(c, c) for c in X_vif.columns]
    vif_data["Centered_VIF"] = [
        variance_inflation_factor(X_vif_const.values, i + 1)
        for i in range(len(macro_test_cols))
    ]
    
    # Sort VIF ascending for plotting
    vif_data = vif_data.sort_values("Centered_VIF", ascending=True).reset_index(drop=True)

    # 5. Visualization
    fig = plt.figure(figsize=(18, 7.5), dpi=300)
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.2], wspace=0.35) 

    # Correlation Matrix
    ax1 = fig.add_subplot(gs[0])
    corr_matrix = X_vif.corr()
    corr_matrix.columns = [vif_plot_names.get(c, c) for c in corr_matrix.columns]
    corr_matrix.index = [vif_plot_names.get(c, c) for c in corr_matrix.index]

    sns.heatmap(
        corr_matrix, annot=True, fmt=".2f", cmap="Blues", square=True, 
        cbar=False, linewidths=0.5, linecolor="white", ax=ax1,
        annot_kws={"size": 10, "weight": "bold"}
    )
    ax1.set_title("1. Pearson Correlation Matrix", fontweight="bold", pad=15, fontsize=11)
    ax1.tick_params(axis='x', rotation=0, labelsize=7.5)
    ax1.tick_params(axis='y', rotation=0, labelsize=10)

    # VIF Bar Chart
    ax2 = fig.add_subplot(gs[1])
    vif_colors = ["#d62728" if v >= 10 else "#ff7f0e" if v >= 5 else "#1f77b4" for v in vif_data["Centered_VIF"]]

    bars = ax2.barh(vif_data["Variable"], vif_data["Centered_VIF"], color=vif_colors, height=0.6)
    ax2.axvline(5, color="#ff7f0e", linestyle="--", linewidth=1.2, label="Moderate Threshold (VIF = 5.0)")
    ax2.axvline(10, color="#d62728", linestyle=":", linewidth=1.5, label="Critical Threshold (VIF = 10.0)")

    for bar in bars:
        width = bar.get_width()
        ax2.text(width + 0.3, bar.get_y() + bar.get_height()/2, f"{width:.2f}", 
                 va="center", ha="left", fontweight="bold", fontsize=9)

    ax2.set_title("2. Variance Inflation Factor (Centered VIF)", fontweight="bold", pad=15, fontsize=11)
    ax2.set_xlabel("VIF Value", fontweight="bold")
    ax2.legend(loc="lower right", frameon=False, fontsize=9) 
    remove_chart_spines(ax2)

    plt.tight_layout()
    output_filename = "panel_vif_macro_diagnostics.png"
    plt.savefig(output_filename, dpi=300, bbox_inches="tight")
    plt.close()
    
    logger.info(f"Dashboard saved successfully: '{output_filename}'")
    logger.info("STAGE 1 COMPLETED.")

if __name__ == "__main__":
    main()